## Introduction

In this notebook you'll be working with behavioral data collected from two experiments in which two mice foraged for food in an environment with three patches whose reward rates changed dynamically over time.

The experiments consist of three periods each:

1. "presocial", in which each mouse was in the environment alone for 3-4 days.
2. "social", in which both mice were in the environment together for 2 weeks.
3. "postsocial", in which each mouse was in the environment alone again for 3-4 days.

The goal of the experiment was to understand how mouse behavior changes as they learn to forage for food in the environment, and how their behavior differs between social vs. solo settings.

## Set up

### Prerequisities

1. [VSCode](https://code.visualstudio.com/download) 

    - re: forks, e.g. Cursor or Windsurf: Cursor does support remote tunneling and is fine to use, Windsurf and most others will NOT WORK as they don't support remote tunneling

    - Ensure you have the 'Python', 'Jupyter', 'Data Wrangler', 'Remote - SSH', and 'Remote - Tunnels' VSCode extensions installed.

2. HPC account credentials:

    - tricentre0
        - tgKOfI*741

    - tricentre1
        - l2b£l6S$98

    - tricentre2
        - p[p9K1b(05
    
    - tricentre3
        - g9-21S&q69

    - tricentre4
        - 5Zh.5a0Js=
    
    - tricentre5
        - NM5@x24<i3
    
    - tricentre6
        - 90+£9Mh21<
    
    - tricentre7
        - m3Z}$wT165
    
    - tricentre8
        - 7ez04]Q7m4
    
    - tricentre9
        - 3Nli;F6Ri9


### Connection Instructions

1. Within a VSCode terminal, ssh into hpc-gw2: 

    ```bash
    ssh -J <tricentre_account>@ssh.swc.ucl.ac.uk <tricentre_account>@hpc-gw2
    ``` 
    (you will be prompted to enter the account's password, twice)

2. In the same terminal, start a remote tunnel on a hpc compute node
    ```bash
    srun --time=05:00:00 --gres=gpu:1 --partition=gpu_lowp --ntasks 16 --mem 64G --pty /usr/bin/bash -i  # connect to compute node
    code tunnel # start tunnel on compute node
    ```
    and then follow the instructions to grant yourself access to this tunnel (typically via GitHub)

3. Keep this instance of VSCode open, and open a new VSCode instance. In this new VSCode instance, open the command palette, and run the 'Remote-Tunnels: Connect to Tunnel' command. After running this from the command palette, you should see an option to click on the tunnel you just opened in the previous step.

4. After you've connected to the tunnel, in the VSCode file explorer, ensure you can access `/ceph/aeon/aeon/tricentre_hackathon_2025` (the base directory for this project). 

5. Within the 'Extensions' tab in VsCode, ensure you sync your locally installed extensions to this remote instance of VSCode, to ensure you can use Python and Jupyer.

6. Create a copy of this notebook (`tricentre_hackathon.ipynb`), rename and save it, then open it in VsCode, select the 'default' environment in the kernel picker, and try running the first code cell (with all the `import`s)

### Experiment Overview

<img src="./tricentre_hackathon_assets/example_foraging_over_blocks.png" width="100%">

<img src="./tricentre_hackathon_assets/social02_env_protocol.png" width="100%">

In [1]:
from IPython.display import HTML

# Construct an iframe that uses srcdoc to embed Mermaid, escaping single quotes in the diagram init config
html = """
<iframe
  srcdoc='<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <script src="https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js"></script>
  <script>mermaid.initialize({ startOnLoad: true, theme: "dark" });</script>
</head>
<body>
  <div class="mermaid">
    %%{init: {&#39;theme&#39;: &#39;dark&#39;}}%%
    gantt
      title Social0.2
      dateFormat  YYYY-MM-DD

      section Aeon3
      BAA-1104045             :2024-01-31, 2024-02-03
      Clean                   :2024-02-04, 2024-02-05
      BAA-1104047             :2024-02-05, 2024-02-08
      Clean                   :2024-02-08, 2024-02-09
      Tube Test               :2024-02-09, 2024-02-10
      BAA-1104045 + BAA-1104047:2024-02-09, 2024-02-23
      Clean                   :2024-02-23, 2024-02-24
      BAA-1104045             :2024-02-25, 2024-02-28
      Clean                   :2024-02-28, 2024-02-29
      BAA-1104047             :2024-02-28, 2024-03-02

      section Aeon4
      BAA-1104048             :2024-01-31, 2024-02-03
      Clean                   :2024-02-04, 2024-02-05
      BAA-1104049             :2024-02-05, 2024-02-08
      Clean                   :2024-02-08, 2024-02-09
      Tube Test               :2024-02-09, 2024-02-10
      BAA-1104048 + BAA-1104049:2024-02-09, 2024-02-23
      Clean                   :2024-02-23, 2024-02-24
      BAA-1104048             :2024-02-25, 2024-02-28
      Clean                   :2024-02-28, 2024-02-29
      BAA-1104049             :2024-02-28, 2024-03-02
  </div>
</body>
</html>'
  style="width:100%; height:1000px; border:0;"
></iframe>
"""

display(HTML(html))


### Python Imports

In [27]:
%load_ext autoreload
%autoreload 2
# %flow mode reactive

import datetime
import sys
import os
import warnings
from IPython.display import display, Markdown
from pathlib import Path
from typing import Any, Tuple, List, Dict

import einops
import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objs as go
import statsmodels.api as sm
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm

import datajoint as dj
dj.config['database.host'] = "aeon-db2"
dj.config['database.user'] = "aeon-tri2025"
dj.config['database.password'] = "hackathon-tri2025"
from aeon.dj_pipeline.analysis.block_analysis import *

from utils import load_all_patch_data, load_all_foraging_bouts, load_all_position_data

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data Overview & Example Analyses

You'll be working with data that contains: 

1. The full-pose position and ID labels of each mouse in the environment.
2. Details on the activity of each mouse in a patch, per block (e.g. time spent in patch, distance foraged in patch, food pellets accumulated in patch, and running and overall patch preference based on these metrics).
3. Details on the patch properties, per block (e.g. patch location, patch reward rate).

For simplicitly, excluded from this example dataset are the continuous raw video (9 cameras), audio (2 microphones), rfid data (9 readers), and weight data (weight scale in the nest) recorded during the experiments.

Some example analyses you could take on are:

1. Compare patch preference between presocial and postsocial periods, social and solo periods, and/or between experiments.

2. Using the already computed foraging bout data, or through creating a simple function to determine some other behavior (e.g. social interactions, given relative position data) see at what times/conditions this behavior occurs across experiment periods.

3. Use an RL approach to learn an animal's policy, or use an inverse RL approach to infer an animal's reward function, where state and state features are defined e.g. by position, pose, patch properties, etc.

Below you'll see some starter examples to load and visualize the data.

In [4]:
"""Set some constants."""

cm2px = 5.2  # 1 cm = 5.2 px roughly in aeon arenas
light_off, light_on = 7, 20  # arena lights off from 7am to 8pm, on from 8pm to 7am

aeon3_coordinates = {
    "NestRegion": [
        (1321, 486),
        (1231, 485),
        (1231, 591),
        (1323, 583),
    ],
    "Patch1Region": [
        (899, 535),
        (922, 533),
        (921, 554),
        (899, 554),
    ],
    "Patch2Region": [
        (610, 709),
        (628, 718),
        (617, 739),
        (600, 730),
    ],
    "Patch3Region": [
        (590, 371),
        (608, 360),
        (619, 380),
        (601, 392),
    ],
    "ArenaCenter": (708, 543),
    "ArenaInnerRadius": 505,
    "ArenaOuterRadius": 531,
}

aeon4_coordinates = {
    "NestRegion": [
        (1273, 518),
        (1183, 517),
        (1183, 623),
        (1275, 615),
    ],
    "Patch1Region": [
        (861, 558),
        (884, 556),
        (883, 577),
        (861, 577),
    ],
    "Patch2Region": [
        (577, 723),
        (595, 732),
        (584, 753),
        (567, 744),
    ],
    "Patch3Region": [
        (565, 394),
        (583, 383),
        (594, 403),
        (576, 415),
    ],
    "ArenaCenter": (680, 565),
    "ArenaInnerRadius": 487,
    "ArenaOuterRadius": 515,
}

In [ ]:
"""Set experiment timelines."""

# For testing purposes, every period is 2 days long
# experiments = [
#     {
#         "name": "social0.2-aeon3", 
#         "presocial_start": "2024-02-01 07:00:00", 
#         "presocial_end": "2024-02-03 07:00:00", 
#         "social_start": "2024-02-10 07:00:00", 
#         "social_end": "2024-02-12 07:00:00", 
#         "postsocial_start": "2024-02-27 07:00:00", 
#         "postsocial_end": "2024-02-29 07:00:00"
#     },
#     {
#         "name": "social0.2-aeon4",
#         "presocial_start": "2024-02-01 07:00:00",
#         "presocial_end": "2024-02-03 07:00:00",
#         "social_start": "2024-02-10 07:00:00",
#         "social_end": "2024-02-12 07:00:00",
#         "postsocial_start": "2024-02-27 07:00:00",
#         "postsocial_end": "2024-02-29 07:00:00"
#     },
# ]


# Full experiment timelines: *nb* if loading all full exp data takes a long time, use the shorter periods above for testing
experiments = [
    {
        "name": "social0.2-aeon3", 
        "presocial_start": "2024-01-31 11:00:00", 
        "presocial_end": "2024-02-08 15:00:00", 
        "social_start": "2024-02-09 16:00:00", 
        "social_end": "2024-02-23 13:00:00", 
        "postsocial_start": "2024-02-25 17:00:00", 
        "postsocial_end": "2024-03-02 14:00:00"
    },
    {
        "name": "social0.2-aeon4",
        "presocial_start": "2024-01-31 11:00:00",
        "presocial_end": "2024-02-08 15:00:00",
        "social_start": "2024-02-09 17:00:00",
        "social_end": "2024-02-23 12:00:00",
        "postsocial_start": "2024-02-25 18:00:00",
        "postsocial_end": "2024-03-02 13:00:00"
    },
]

### Position data

In [28]:
"""Load all centroid data."""

position_data_dict = load_all_position_data(experiments)

In [ ]:
"""Visualize centroid data."""

exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Extract the position DataFrame
df = position_data_dict[exp_name][period]

# Display the first few rows
md = (
    f"### Position Data — {exp_name}, {period}\n"
    "Columns:\n"
    "- `time (index)`: Timestamp of the position data\n"
    "- `experiment_name`: Name of the experiment\n"
    "- `identity_name`: Name of the tracked identity (e.g., mouse)\n"
    "- `identity_likelihood`: Likelihood score for the tracked identity\n"
    "- `x`: X coordinate of the centroid position (in pixels)\n"
    "- `y`: Y coordinate of the centroid position (in pixels)\n"
    "- `likelihood`: Likelihood score for the (x,y) position\n"
    "- `anchor_part`: Anchor part used for tracking (centroid of the mouse)\n"
    "- `period`: Period of the experiment (e.g., presocial, social, postsocial)\n\n"
)
display(Markdown(md))
display(df.head())

### Position Data — social0.2-aeon3, social
Columns:
- `time`: Timestamp of the position data
- `experiment_name`: Name of the experiment
- `identity_name`: Name of the tracked identity (e.g., mouse)
- `likelihood`: Likelihood score for the (x,y) position
- `x`: X coordinate of the centroid (spine2) position (in pixels)
- `y`: Y coordinate of the centroid (spine2) position (in pixels)
- `period`: Period of the experiment (e.g., presocial, social, postsocial)



,experiment_name,identity_name,likelihood,x,y,period
time,,,,,,
2024-02-10 07:00:00.000,social0.2-aeon3,BAA-1104045,0.815682,1299.434082,507.100220,social
2024-02-10 07:00:00.140,social0.2-aeon3,BAA-1104047,0.727575,1309.625732,499.341278,social
2024-02-10 07:00:00.140,social0.2-aeon3,BAA-1104045,0.727575,1299.518799,507.131927,social
2024-02-10 07:00:00.160,social0.2-aeon3,BAA-1104045,0.727724,1299.511841,507.128998,social
2024-02-10 07:00:00.160,social0.2-aeon3,BAA-1104047,0.727724,1309.630615,499.340668,social


In [30]:
"""Plot a heatmap for a given experiment and period."""

# Choose which experiment and period to visualize
exp_name = experiments[0]["name"]
period = "social"  # options: "presocial", "social", or "postsocial"

# Load position data and drop any missing values
df = position_data_dict[exp_name][period]
df = df.dropna(subset=['x', 'y']).reset_index(drop=True)

# Filter out occasional erroneous positions that fall outside the arena/nest box
#   - Arena: circle with padded outer radius
outer_radius = float(aeon3_coordinates["ArenaOuterRadius"]) + 10
center_x, center_y = map(float, aeon3_coordinates["ArenaCenter"])
dist2 = (df['x'] - center_x)**2 + (df['y'] - center_y)**2
inside_arena = dist2 <= outer_radius**2
#   - Nest box: axis-aligned bounding box of nest region, with padding
nest_pts = aeon3_coordinates["NestRegion"]
x_pts = [float(x) for x, _ in nest_pts]
y_pts = [float(y) for _, y in nest_pts]
nx_min, nx_max = min(x_pts) - 10, max(x_pts) + 10
ny_min, ny_max = min(y_pts) - 10, max(y_pts) + 10
inside_nest = (
    df['x'].between(nx_min, nx_max) &
    df['y'].between(ny_min, ny_max)
)
# Filter data
df = df[inside_arena | inside_nest]

# Prepare x and y arrays for 2D histogram
x = df['x'].values
y = df['y'].values

# Bin the points into a 2D histogram
bins = 100
counts, xedges, yedges = np.histogram2d(x, y, bins=bins)

# Apply log(1 + count) transform to avoid high-density saturation
counts_log = np.log1p(counts)

# Compute bin centers from edge arrays
xcenters = (xedges[:-1] + xedges[1:]) / 2
ycenters = (yedges[:-1] + yedges[1:]) / 2

# Clip color scale at the 99th percentile to preserve contrast
vmax = np.nanpercentile(counts_log, 99)

# Create Plotly heatmap figure
fig = go.Figure(go.Heatmap(
    x=xcenters,
    y=ycenters,
    z=counts_log.T,          # transpose so x→columns, y→rows
    colorscale='Turbo',      # vivid, high-contrast palette
    zmin=0,
    zmax=vmax,
    colorbar=dict(
        title='log(count+1)',
        thickness=20,
        len=0.75,
    )
))

# Polish axes, layout, and styling
fig.update_layout(
    title=f"Position heatmap for {exp_name!r} during “{period}”",
    width=700,
    height=700,
    margin=dict(l=60, r=20, t=60, b=60),
    xaxis=dict(
        title='X position',
        showgrid=False,
        zeroline=False,
        showline=True,
        ticks='outside'
    ),
    yaxis=dict(
        title='Y position',
        scaleanchor="x",
        scaleratio=1,
        showgrid=False,
        zeroline=False,
        showline=True,
        ticks='outside'
    ),
    template='plotly_white'
)

# Render a static plot
fig.show(config={'staticPlot': True})


In [11]:
"""Visualize full pose data."""

# *NB* -- This data is typically over 100 mb per mouse-hour, 
# *NB* -- and may take over 1 minute to load per mouse-hour!

key = {"experiment_name": "social0.2-aeon3"}
period_start = "2024-02-09 16:00:00"
period_end = "2024-02-09 16:59:00"
chunk_restriction = acquisition.create_chunk_restriction(
    key["experiment_name"], period_start, period_end
)
full_pose_data = (
    tracking.SLEAPTracking.PoseIdentity.proj("identity_name", "identity_likelihood")
    * tracking.SLEAPTracking.Part
    & key
    & chunk_restriction
).fetch(format="frame")
full_pose_data = full_pose_data.droplevel(
    level=[
        "device_serial_number",
        "spinnaker_video_source_install_time",
        "tracking_paramset_id",
        "identity_idx"
    ]
)
display(full_pose_data)

identity_name  \
experiment_name chunk_start         part_name                 
social0.2-aeon3 2024-02-09 16:07:32 head        BAA-1104045   
                                    left_ear    BAA-1104045   
                                    nose        BAA-1104045   
                                    right_ear   BAA-1104045   
                                    spine1      BAA-1104045   
                                    spine2      BAA-1104045   
                                    spine3      BAA-1104045   
                                    spine4      BAA-1104045   
                                    head        BAA-1104047   
                                    left_ear    BAA-1104047   
                                    nose        BAA-1104047   
                                    right_ear   BAA-1104047   
                                    spine1      BAA-1104047   
                                    spine2      BAA-1104047   
                                    spine3      BAA-1104047   
                                    spine4      BAA-1104047   

                                                                             identity_likelihood  \
experiment_name chunk_start         part_name                                                      
social0.2-aeon3 2024-02-09 16:07:32 head       [0.79847, 0.5827296, 0.6770189, 0.77968746, 0....   
                                    left_ear   [0.79847, 0.5827296, 0.6770189, 0.77968746, 0....   
                                    nose       [0.79847, 0.5827296, 0.6770189, 0.77968746, 0....   
                                    right_ear  [0.79847, 0.5827296, 0.6770189, 0.77968746, 0....   
                                    spine1     [0.79847, 0.5827296, 0.6770189, 0.77968746, 0....   
                                    spine2     [0.79847, 0.5827296, 0.6770189, 0.77968746, 0....   
                                    spine3     [0.79847, 0.5827296, 0.6770189, 0.77968746, 0....   
                                    spine4     [0.79847, 0.5827296, 0.6770189, 0.77968746, 0....   
                                    head       [nan, nan, nan, nan, nan, nan, nan, nan, nan, ...   
                                    left_ear   [nan, nan, nan, nan, nan, nan, nan, nan, nan, ...   
                                    nose       [nan, nan, nan, nan, nan, nan, nan, nan, nan, ...   
                                    right_ear  [nan, nan, nan, nan, nan, nan, nan, nan, nan, ...   
                                    spine1     [nan, nan, nan, nan, nan, nan, nan, nan, nan, ...   
                                    spine2     [nan, nan, nan, nan, nan, nan, nan, nan, nan, ...   
                                    spine3     [nan, nan, nan, nan, nan, nan, nan, nan, nan, ...   
                                    spine4     [nan, nan, nan, nan, nan, nan, nan, nan, nan, ...   

                                               sample_count  \
experiment_name chunk_start         part_name                 
social0.2-aeon3 2024-02-09 16:07:32 head             112784   
                                    left_ear         112784   
                                    nose             112784   
                                    right_ear        112784   
                                    spine1           112784   
                                    spine2           112784   
                                    spine3           112784   
                                    spine4           112784   
                                    head             112481   
                                    left_ear         112481   
                                    nose             112481   
                                    right_ear        112481   
                                    spine1           112481   
                                    spine2           112481   
                                    spine3           112481   
                                    spi

### Patch data

In [12]:
"""Load all patch data."""

patch_info_dict, subject_patch_data_dict, subject_patch_pref_dict = load_all_patch_data(experiments)

In [13]:
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Extract example DataFrames
df1 = patch_info_dict[exp_name][period]
df2 = subject_patch_data_dict[exp_name][period]
df3 = subject_patch_pref_dict[exp_name][period]

# Display the first few rows of each
md = (
    f"### Patch Info — {exp_name}, {period}\n"
    "#### Each row details the patch properties for a given block.\n"
    "Columns:\n"
    "- `experiment_name`: Name of the experiment\n"
    "- `period`: Period of the experiment (e.g., presocial, social, postsocial)\n"
    "- `block_start`: Start time of the block\n"
    "- `patch_name`: Name of the patch\n"
    "- `wheel_timestamps`: Sampled wheel timestamps\n"
    "- `patch_rate`: Exponential distribution 'rate' parameter "
    "(`1 / rate = mean`, so the lower the rate, the 'harder' the patch)\n"
    "- `patch_offset`: Exponential distribution 'offset' parameter "
    "(lower bound of the distribution)\n"
)
display(Markdown(md))
display(df1.head())

md = (
    f"### Subject Patch Data — {exp_name}, {period}\n"
    "#### Each row details a subject's interaction with a specific patch in a specific block.\n"
    "Columns:\n"
    "- `experiment_name`: Name of the experiment\n"
    "- `period`: Period of the experiment (e.g., presocial, social, postsocial)\n"
    "- `block_start`: Start time of the block\n"
    "- `patch_name`: Name of the patch\n"
    "- `subject_name`: Name of the subject\n"
    "- `in_patch_timestamps`: Array of timestamps when the subject was in the patch\n"
    "- `in_patch_time`: Total time spent in the patch (in seconds)\n"
    "- `in_patch_rfid_timestamps`: Array of timestamps when the subject was detected " 
    "in the patch via RFID\n"
    "- `pellet_counts`: Number of pellets delivered\n"
    "- `pellet_timestamps`: Array of timestamps when pellets were delivered\n"
    "- `patch_threshold`: Array of 'wheel distance spun' thresholds (in cm) "
    "for each delivered pellet\n"
    "- `wheel_cumsum_distance_travelled`: Cumulative distance that the "
    "foraging patch wheel was spun (in cm)\n"
)
display(Markdown(md))
display(df2.head())

md = (
    f"### Subject Patch Preference — {exp_name}, {period}\n"
    "#### Each row details the preference of a subject for a specific patch in a specific block.\n"
    "Columns:\n"
    "- `experiment_name`: Name of the experiment\n"
    "- `period`: Period of the experiment (e.g., presocial, social, postsocial)\n"
    "- `block_start`: Start time of the block\n"
    "- `patch_name`: Name of the patch\n"
    "- `subject_name`: Name of the subject\n"
    "- `cumulative_preference_by_time`: Array of relative patch preference values (0-1) " 
    "by time spent in patch, relative to the final preference at the end of the block.\n"
    "- `cumulative_preference_by_wheel`: Array of relative patch preference values (0-1) "
    "by patch wheel distance spun, relative to the final preference at the end of the block.\n"
    "- `running_preference_by_time`: Array of relative patch preference values (0-1) "
    "by time spent in patch up to the current timepoint.\n"
    "- `running_preference_by_wheel`: Array of relative patch preference values (0-1) "
    "by patch wheel distance spun up to the current timepoint.\n"
    "- `final_preference_by_time`: Final scalar relative patch preference value (0-1) "
    "by time spent in patch at the end of the block.\n"
    "- `final_preference_by_wheel`: Final scalar relative patch preference value (0-1) "
    "by patch wheel distance spun at the end of the block.\n"
)
display(Markdown(md))
display(df3.head())

### Patch Info — social0.2-aeon3, social
#### Each row details the patch properties for a given block.
Columns:
- `experiment_name`: Name of the experiment
- `period`: Period of the experiment (e.g., presocial, social, postsocial)
- `block_start`: Start time of the block
- `patch_name`: Name of the patch
- `wheel_timestamps`: Sampled wheel timestamps
- `patch_rate`: Exponential distribution 'rate' parameter (`1 / rate = mean`, so the lower the rate, the 'harder' the patch)
- `patch_offset`: Exponential distribution 'offset' parameter (lower bound of the distribution)


,experiment_name,period,block_start,patch_name,wheel_timestamps,patch_rate,patch_offset
0,social0.2-aeon3,social,2024-02-10 08:09:48.001984,Patch1,"[2024-02-10T08:09:48.020000000, 2024-02-10T08:...",0.002,75.0
1,social0.2-aeon3,social,2024-02-10 08:09:48.001984,Patch2,"[2024-02-10T08:09:48.020000000, 2024-02-10T08:...",0.002,75.0
2,social0.2-aeon3,social,2024-02-10 08:09:48.001984,Patch3,"[2024-02-10T08:09:48.020000000, 2024-02-10T08:...",0.002,75.0
3,social0.2-aeon3,social,2024-02-10 09:25:24.001984,Patch1,"[2024-02-10T09:25:24.020000000, 2024-02-10T09:...",0.010,75.0
4,social0.2-aeon3,social,2024-02-10 09:25:24.001984,Patch2,"[2024-02-10T09:25:24.020000000, 2024-02-10T09:...",0.010,75.0


### Subject Patch Data — social0.2-aeon3, social
#### Each row details a subject's interaction with a specific patch in a specific block.
Columns:
- `experiment_name`: Name of the experiment
- `period`: Period of the experiment (e.g., presocial, social, postsocial)
- `block_start`: Start time of the block
- `patch_name`: Name of the patch
- `subject_name`: Name of the subject
- `in_patch_timestamps`: Array of timestamps when the subject was in the patch
- `in_patch_time`: Total time spent in the patch (in seconds)
- `in_patch_rfid_timestamps`: Array of timestamps when the subject was detected in the patch via RFID
- `pellet_counts`: Number of pellets delivered
- `pellet_timestamps`: Array of timestamps when pellets were delivered
- `patch_threshold`: Array of 'wheel distance spun' thresholds (in cm) for each delivered pellet
- `wheel_cumsum_distance_travelled`: Cumulative distance that the foraging patch wheel was spun (in cm)


,experiment_name,period,block_start,patch_name,subject_name,in_patch_timestamps,in_patch_time,in_patch_rfid_timestamps,pellet_count,pellet_timestamps,patch_threshold,wheel_cumsum_distance_travelled
0,social0.2-aeon3,social,2024-02-10 08:09:48.001984,Patch1,BAA-1104045,"[2024-02-10T08:14:18.800000000, 2024-02-10T08:...",373.44,"[2024-02-10T08:14:20.800320000, 2024-02-10T08:...",7,"[2024-02-10T08:17:02.383488000, 2024-02-10T08:...","[245.58469465309585, 643.0188679333108, 295.35...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,social0.2-aeon3,social,2024-02-10 08:09:48.001984,Patch1,BAA-1104047,"[2024-02-10T08:17:43.840000000, 2024-02-10T08:...",128.76,"[2024-02-10T08:33:08.098656000, 2024-02-10T08:...",3,"[2024-02-10T08:33:16.726496000, 2024-02-10T08:...","[923.3001666426386, 285.1629112804651, 227.141...","[-0.0, -0.006136297681432978, 0.00153407442035..."
2,social0.2-aeon3,social,2024-02-10 08:09:48.001984,Patch2,BAA-1104045,"[2024-02-10T08:13:44.160000000, 2024-02-10T08:...",468.00,"[2024-02-10T08:16:23.416768000, 2024-02-10T08:...",10,"[2024-02-10T08:28:25.639488000, 2024-02-10T08:...","[134.53653581217822, 453.0393906120812, 157.52...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,social0.2-aeon3,social,2024-02-10 08:09:48.001984,Patch2,BAA-1104047,"[2024-02-10T08:11:14.460000000, 2024-02-10T08:...",144.72,"[2024-02-10T08:11:16.890432000, 2024-02-10T08:...",2,"[2024-02-10T08:34:17.066496000, 2024-02-10T08:...","[793.663145142955, 118.4609162564764]","[-0.0, -0.00153407442035558, 0.001534074420359..."
4,social0.2-aeon3,social,2024-02-10 08:09:48.001984,Patch3,BAA-1104045,"[2024-02-10T08:09:50.400000000, 2024-02-10T08:...",533.64,"[2024-02-10T08:15:19.524576000, 2024-02-10T08:...",7,"[2024-02-10T08:17:46.915488000, 2024-02-10T08:...","[525.3902337548575, 691.3086599739768, 1926.70...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


### Subject Patch Preference — social0.2-aeon3, social
#### Each row details the preference of a subject for a specific patch in a specific block.
Columns:
- `experiment_name`: Name of the experiment
- `period`: Period of the experiment (e.g., presocial, social, postsocial)
- `block_start`: Start time of the block
- `patch_name`: Name of the patch
- `subject_name`: Name of the subject
- `cumulative_preference_by_time`: Array of relative patch preference values (0-1) by time spent in patch, relative to the final preference at the end of the block.
- `cumulative_preference_by_wheel`: Array of relative patch preference values (0-1) by patch wheel distance spun, relative to the final preference at the end of the block.
- `running_preference_by_time`: Array of relative patch preference values (0-1) by time spent in patch up to the current timepoint.
- `running_preference_by_wheel`: Array of relative patch preference values (0-1) by patch wheel distance spun up to the current timepoint.
- `final_preference_by_time`: Final scalar relative patch preference value (0-1) by time spent in patch at the end of the block.
- `final_preference_by_wheel`: Final scalar relative patch preference value (0-1) by patch wheel distance spun at the end of the block.


,experiment_name,period,block_start,patch_name,subject_name,cumulative_preference_by_wheel,cumulative_preference_by_time,running_preference_by_time,running_preference_by_wheel,final_preference_by_wheel,final_preference_by_time
0,social0.2-aeon3,social,2024-02-10 08:09:48.001984,Patch1,BAA-1104045,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.297947,0.271577
1,social0.2-aeon3,social,2024-02-10 08:09:48.001984,Patch1,BAA-1104047,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.155628,0.170358
2,social0.2-aeon3,social,2024-02-10 08:09:48.001984,Patch2,BAA-1104045,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.329502,0.340344
3,social0.2-aeon3,social,2024-02-10 08:09:48.001984,Patch2,BAA-1104047,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.080129,0.191474
4,social0.2-aeon3,social,2024-02-10 08:09:48.001984,Patch3,BAA-1104045,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.372552,0.388079


In [14]:
"""Visualize patch preference by reward rate over blocks per subject"""

patch_rate_map = {0.01: "high", 0.0033: "medium", 0.002: "low"}

# Using 'patch_info', group by block_start, then
# Filter by blocks in which total pellet_count across patches > 2
# For all of these blocks, get the final patch preference by wheel for each patch
# Then map these patches to (high, medium low reward) patches based on patch_rate (inverse of patch rate determines reward level)
# Then plot this final patch preference for each patch type per block for all blocks (three dots per block, x-axis in blocks)

block_pellet_counts = df2.groupby('block_start')['pellet_count'].sum()
filt_block_starts = block_pellet_counts[block_pellet_counts > 2].index.tolist()

# This DataFrame contains all relevant (block, patch, subject) combinations
filt_blocks_df = df3[df3['block_start'].isin(filt_block_starts)].reset_index(drop=True)

df1_rates = df1[['block_start', 'patch_name', 'patch_rate']].drop_duplicates()

# Iterate over subjects
unique_subjects = filt_blocks_df['subject_name'].unique()
for subject in unique_subjects:
    # Filter data for the current subject
    subj_data = filt_blocks_df[filt_blocks_df['subject_name'] == subject]

    # Merge with patch rates
    merged_df_subject = pd.merge(
        subj_data, df1_rates, on=['block_start', 'patch_name'], how='left'
    )

    # Map patch_rate_map
    plot_df_subject = merged_df_subject.copy()
    plot_df_subject['reward_rate'] = plot_df_subject['patch_rate'].map(patch_rate_map)
    
    # Ensure subject data
    if plot_df_subject.empty:
        print(f"No data to plot for subject: {subject} after filtering and mapping reward rates.")
        continue

    # Prettify block start time
    plot_df_subject['block_start'] = pd.to_datetime(plot_df_subject['block_start'])
    plot_df_subject['block_start_str'] = plot_df_subject['block_start'].dt.round("s").dt.strftime('%Y-%m-%d %H:%M:%S')

    # Plot it
    fig = px.scatter(
        plot_df_subject,
        x='block_start_str',
        y='final_preference_by_wheel',
        color='reward_rate',
        category_orders={"reward_rate": ["high", "medium", "low"]},
        color_discrete_map={"high": "orange", "medium": "gold", "low": "blue"},
        # Update title to include the subject name
        title=f'Final Patch Preference by Reward Rate per Block - Subject: {subject}',
        labels={
            'block_start_str': 'Block Start Time',
            # Y-axis label no longer says "Avg. across Subjects"
            'final_preference_by_wheel': 'Final Preference by Wheel',
            'reward_rate': 'Reward Rate'
        },
        hover_data={'patch_name': True} # Optionally show patch_name on hover
    )
        
    fig.update_xaxes(type='category') 
    fig.update_layout(xaxis_title='Block Start Time', yaxis_title='Final Preference by Wheel')
    
    print(f"Displaying plot for subject: {subject}")
    fig.show()

Displaying plot for subject: BAA-1104045


Displaying plot for subject: BAA-1104047


In [15]:
"""Visualize wheel spun distance per subject per patch."""

exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Get the subject patch data
df = subject_patch_data_dict[exp_name][period]

# Ensure data is sorted and ready
dt_seconds = 0.02
subjects = sorted(df['subject_name'].unique())
patches = sorted(df['patch_name'].unique())

fig = go.Figure()

# Build each trace (continuous + Δ>0.5 downsample)
for (subject, patch), grp in df.groupby(['subject_name', 'patch_name']):
    grp = grp.sort_values('block_start')
    total_n = sum(len(a) for a in grp.wheel_cumsum_distance_travelled)

    times = np.empty(total_n, dtype='datetime64[ns]')
    dists = np.empty(total_n, dtype=float)
    idx, offset = 0, 0.0

    for bs_val, arr in zip(grp.block_start, grp.wheel_cumsum_distance_travelled):
        arr = np.asarray(arr)
        n = arr.size
        offs = (np.arange(n) * dt_seconds * 1e9).astype('timedelta64[ns]')
        times[idx:idx+n] = np.datetime64(bs_val) + offs
        dists[idx:idx+n] = arr + offset
        offset += arr[-1]
        idx += n

    # Downsample by change > 0.5
    diffs = np.abs(np.diff(dists, prepend=dists[0]))
    mask = diffs > 0.5
    mask[0] = True

    fig.add_trace(
        go.Scatter(
            x=times[mask],
            y=dists[mask] / 100,  # convert to meters
            mode='lines',
            name=f"{subject} — {patch}",
            line=dict(width=1.5)
        )
    )

# Styling (match px.timeline aesthetic)
fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    showline=True,
    ticks='outside',
    showticklabels=True
)

fig.update_yaxes(
    title_text="Distance spun on wheel (m)",
    showgrid=False,
    zeroline=False,
    showline=True,
    ticks='outside',
    showticklabels=True
)

fig.update_layout(
    template="simple_white",
    plot_bgcolor="white",
    showlegend=True
)

fig.show(config={'staticPlot': True})

In [16]:
"""Load all foraging bouts data."""

foraging_data_dict = load_all_foraging_bouts(experiments)

In [17]:
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Extract the foraging DataFrame
df = foraging_data_dict[exp_name][period]

# Display the first few rows
md = (
    f"### Foraging Data — {exp_name}, {period}\n"
    "Columns:\n"
    "- `experiment_name`: Name of the experiment\n"
    "- `start`: Start time of the foraging bout\n"
    "- `end`: End time of the foraging bout\n"
    "- `n_pellets`: Number of pellets consumed during the bout\n"
    "- `cum_wheel_dist`: Cumulative distance spun on the wheel during the bout (in cm)\n"
    "- `subject`: Subject name\n"
    "- `period`: Period of the experiment (e.g., presocial, social, postsocial)\n\n"
)
display(Markdown(md))
display(df.head())

### Foraging Data — social0.2-aeon3, social
Columns:
- `experiment_name`: Name of the experiment
- `start`: Start time of the foraging bout
- `end`: End time of the foraging bout
- `n_pellets`: Number of pellets consumed during the bout
- `cum_wheel_dist`: Cumulative distance spun on the wheel during the bout (in cm)
- `subject`: Subject name
- `period`: Period of the experiment (e.g., presocial, social, postsocial)



,experiment_name,start,end,n_pellets,cum_wheel_dist,subject,period
0,social0.2-aeon3,2024-02-10 08:14:21.700,2024-02-10 08:19:12.760,2,2394.417105,BAA-1104045,social
1,social0.2-aeon3,2024-02-10 08:19:14.880,2024-02-10 08:25:09.660,5,3244.101040,BAA-1104045,social
2,social0.2-aeon3,2024-02-10 08:27:50.900,2024-02-10 08:33:35.960,5,2791.351191,BAA-1104045,social
3,social0.2-aeon3,2024-02-10 08:33:12.580,2024-02-10 08:35:43.280,3,245.000889,BAA-1104047,social
4,social0.2-aeon3,2024-02-10 08:38:59.840,2024-02-10 08:44:09.020,6,1871.040003,BAA-1104045,social


In [18]:
"""Visualize foraging bouts per subject across all patches."""

exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Get the foraging data
df = foraging_data_dict[exp_name][period]

# Ensure correct types
df['start'] = pd.to_datetime(df['start'])
df['end'] = pd.to_datetime(df['end'])

# Plot
subjects = sorted(df['subject'].unique())

fig = px.timeline(
    df,
    x_start="start",
    x_end="end",
    y="subject",
    hover_data=["n_pellets", "cum_wheel_dist"],
    category_orders={"subject": subjects}
)

# Styling
fig.update_traces(
    opacity=1,
    marker_color="#555555",
    marker_line_color="#555555",
    marker_line_width=1.5
)

fig.update_layout(
    template='simple_white',
    plot_bgcolor='white',
    margin=dict(l=150, r=20, t=20, b=20),
    height=max(100, len(subjects) * 25 + 50),
    xaxis=dict(
        showgrid=False, zeroline=False,
        showline=True, ticks='outside', showticklabels=False
    ),
    yaxis=dict(
        showgrid=False, zeroline=False,
        showline=False, ticks='', showticklabels=True,
        title=''
    )
)

fig.show()